In [1]:
import streamlit as st
from utils.pipeline import *

In [2]:
def segment_series(series):
    old_dcms = fetch_dicoms_for_series(series.orthanc_id)
    mask = run_inference_on_scan(old_dcms)
    return mask, old_dcms

In [ ]:

db = TinyDB(DB_PATH)
StudyQuery = Query()
existing_ids = [s["orthanc_study_id"] for s in db]

target_list = ['ML cine']

studies = fetch_studies()
for study_info in studies:
    # if study_info["ID"] in existing_ids:
    #     continue
    
    study = Study(study_info)
    mri_sorter = MRI_Sorter(study)
    series_type_df = mri_sorter.sort_df
    series_list = fetch_series_for_study(study.orthanc_id)


    for series_info in series_list:
        series = Series(series_info)
        if series.series_type is None: 
            series_id = series_info['ID']
            if series_id in series_type_df.index:
                series.series_type = series_type_df.loc[series_id]['Type']
            else:
                series.series_type = 'non-image'

        # if series.dl_orthanc_id  is None:
        
        if any(w in series.description for w in target_list): # probably need to add type here as well
            series.series_orientation = 'SAX'
            mask, old_dcms = segment_series(series)
            new_uid = send_series_to_orthanc(mask, old_dcms, new_description="DL Processed Series")
            new_orthanc_id = get_orthanc_series_data_from_uid(new_uid)
            print(new_uid, new_orthanc_id)
            series.dl_orthanc_id = new_orthanc_id['ID']
            
        if series.dl_orthanc_id and not series.roundel_orthanc_id: 
            # add roundel
            pass
        study.series_list.append(series)
    db.upsert(study.to_dict(), StudyQuery.study_uid == study.uid)
db.close()

/home/tina/Documents/PhD/Project/imageCLASP/clasp/lib/python3.13/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Time: 1.5971 seconds
1.2.826.0.1.3680043.8.498.85224351452979664655986178249851964529 {'ExpectedNumberOfInstances': 32, 'ID': 'ba466fbb-9993dc4f-0993e04e-e0697c12-3fb0e6e3', 'Instances': ['03ccf910-17eeb980-450d3eba-23169922-48e508a2', '0b925ab3-e59f9eba-e8f93de6-5ec688ae-5a85df47', '0c114ce0-c79bb917-e9e6e6f7-c69d533e-5ca80f86', '0e037bc3-44d9379c-c6471644-bee1fd16-de916c63', '14854bf4-8f704eed-b0c6879a-3533067b-a6f6b2d3', '199ccde2-02554e3e-31d219c8-ee167397-cd0702a9', '1f7a475d-0e4fdafc-f120635c-59140dbd-fbc99a70', '29a10447-4e8b3401-4d567133-f6c61f50-bf0ae4a8', '355ad25b-eaf6e2e3-49f9b8ed-8cf7ada1-e5b2d5ed', '3d39789c-02aa3519-5909c818-a947bfdc-8b8c37bb', '3e101eee-0c09e691-a2613ad3-491e6057-a47a642f', '459467a5-3d084c3f-9aec0cfa-da5ed4e3-4b43cb29', '5cab6430-8b66db88-c267c758-c1763684-56e98fd1', '627c904c-af42608c-530d07e1-0dc61925-d578343f', '63cfee2f-8f17e7ab-10e9c41f-d436faa4-8c72d279', '646bcf76-26c0e6ee-4636f592-1b246802-872d43f7', '7d5e0d3c-50506c36-78786181-57795272-9580ac7

In [1]:
new_uid

NameError: name 'new_uid' is not defined